# Week 1 self-study: NumPy and linear algebra recap

### Learning goals
After working through the notebook you should be able to:

- create and inspect NumPy arrays;
- use indexing and slicing;
- distinguish element-wise multiplication from matrix multiplication;
- compute a transpose, dot product, and Euclidean norm;
- solve a small linear system with NumPy;
- connect these operations to image arrays.

If all sections are familiar, skim the examples and complete the checks. If NumPy is new to you, run every cell and modify the examples.

## 1. Setup and images as arrays

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, io

try:
    import oip
    oip_available = True
except Exception as exc:
    oip_available = False
    print('Could not import oip:')
    print(repr(exc))

print('oip available:', oip_available)

In [ ]:
gray = oip.load_gray()
rgb = oip.load_rgb()

print('gray:', gray.shape, gray.dtype, gray.min(), gray.max())
print('rgb :', rgb.shape, rgb.dtype, rgb.min(), rgb.max())

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(gray, cmap='gray')
ax[0].set_title('grayscale')
ax[1].imshow(rgb)
ax[1].set_title('RGB')
for a in ax:
    a.axis('off')
plt.tight_layout()

Three properties are worth checking whenever you receive a new image array:

1. `shape` - how many rows, columns, and channels?
2. `dtype` - how are values represented?
3. numerical range - what do the numbers mean?

## 2. Scalars, vectors, matrices, and array shape

In [ ]:
# Scalar
s = 3.0

# Vector: a one-dimensional NumPy array
x = np.array([1.0, 2.0, 3.0])

# Matrix: a two-dimensional NumPy array
A = np.array([[1.0, 2.0, 0.0],
              [0.0, 1.0, 1.0]])

print('x =', x)
print('x.shape =', x.shape, 'x.ndim =', x.ndim)
print('A =\n', A)
print('A.shape =', A.shape, 'A.ndim =', A.ndim)

A one-dimensional NumPy array has shape `(n,)`. NumPy does not automatically distinguish a row vector from a column vector. When dimensions matter explicitly, reshape it, for example `x.reshape(-1, 1)`.

In [ ]:
x_col = x.reshape(-1, 1)
x_row = x.reshape(1, -1)
print('column shape:', x_col.shape)
print('row shape   :', x_row.shape)

## 3. Indexing and slicing

In [ ]:
M = np.arange(30).reshape(5, 6)
print(M)
print('\nOne value M[2, 3] =', M[2, 3])
print('\nRow 2:', M[2, :])
print('\nColumn 3:', M[:, 3])
print('\nBlock M[1:4, 2:5]:\n', M[1:4, 2:5])

The same syntax has an immediate image-processing interpretation: a two-dimensional slice is a crop.

In [ ]:
h, w = gray.shape[:2]
r0, r1 = h // 4, 3 * h // 4
c0, c1 = w // 4, 3 * w // 4
crop = gray[r0:r1, c0:c1]

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(gray, cmap='gray')
ax[0].set_title('original')
ax[1].imshow(crop, cmap='gray')
ax[1].set_title('slice = crop')
for a in ax:
    a.axis('off')
plt.tight_layout()

### Check 1
Predict the shape of `gray[::2, ::2]` before running the next cell. What operation on the sampling grid does this represent?

In [ ]:
half_samples = gray[::2, ::2]
print('original shape:', gray.shape)
print('new shape     :', half_samples.shape)

**Interpretation:** we retained every second row and every second column. We reduced the number of spatial samples; we did not change the bit depth.

## 4. Arithmetic, data types, and clipping

In [ ]:
# Convert to float before arithmetic so that the numerical behaviour is explicit.
g = gray.astype(np.float64)

brighter = np.clip(g + 40, 0, 255)
darker = np.clip(g - 40, 0, 255)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, im, title in zip(ax, [darker, g, brighter], ['-40', 'original', '+40']):
    a.imshow(im, cmap='gray', vmin=0, vmax=255)
    a.set_title(title)
    a.axis('off')
plt.tight_layout()

With integer images, always think about the allowed value range. Silent overflow/underflow or unintended clipping can change the meaning of an operation.

## 5. Element-wise operations versus matrix multiplication

In [ ]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

print('A * B (element-wise):\n', A * B)
print('\nA @ B (matrix product):\n', A @ B)

These operations answer different mathematical questions:

- `A * B`: values at corresponding positions are multiplied.
- `A @ B`: rows of the first matrix are combined with columns of the second.

In image processing, element-wise operations are common for masks and point operations. Matrix products become important when we represent transformations and optimisation models.

## 6. Dot products and matrix-vector products

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 0.0, -1.0])

print('dot product a . b =', a @ b)

A = np.array([[2.0, 0.0, 1.0],
              [0.0, 1.0, 1.0]])
print('A @ a =', A @ a)

A matrix-vector product `y = A @ x` can be interpreted as a linear mapping from one vector to another. This idea will return in filtering, least squares, and inverse problems.

## 7. Transpose and identity

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])
print('A shape:', A.shape)
print('A.T shape:', A.T.shape)
print('A.T =\n', A.T)
print('\nIdentity matrix:\n', np.eye(3))

## 8. Norms: measuring size and difference

In [ ]:
x = np.array([3.0, 4.0])
print('||x||_2 =', np.linalg.norm(x))

# Image difference: compare original with a mildly quantised version.
g = gray.astype(np.float64)
levels = 16
q = np.round(g / 255 * (levels - 1)) / (levels - 1) * 255
error = q - g
print('Euclidean norm of image error =', np.linalg.norm(error))
print('RMS error =', np.sqrt(np.mean(error**2)))

A norm turns a vector (or a flattened image difference) into one non-negative number. In optimisation we often minimise objectives built from norms.

## 9. Solving a small linear system

In [ ]:
A = np.array([[3.0, 1.0],
              [1.0, 2.0]])
b = np.array([9.0, 8.0])

x = np.linalg.solve(A, b)
print('solution x =', x)
print('check A @ x =', A @ x)
print('b           =', b)

For numerical work, prefer `np.linalg.solve(A, b)` over explicitly forming `np.linalg.inv(A) @ b`. We will revisit linear systems when Newton's method and least squares appear.

We postpone the following until they have a purpose in the course:

- eigenvalues and eigenvectors;
- conditioning;
- positive definiteness;
- gradients and Hessians;
- advanced numerical linear algebra.

You should recognise these terms from previous studies.

## Final self-check

<div class="alert alert-block alert-info">Run the cells below, answer each question, then click <b>Check answers</b>. This is for your own practice &ndash; it is not graded or submitted anywhere.</div>


In [ ]:
# Week 1 recap self-check
QUESTIONS = [['What is the difference between `A * B` and `A @ B` in NumPy?', ['They are identical for arrays', '`*` is element-wise multiplication, while `@` is matrix multiplication', '`*` is matrix multiplication, while `@` is element-wise multiplication', '`@` only works with scalars'], 1], ['An RGB image has shape `(1200, 1600, 3)`. What does the final dimension represent?', ['Bit depth', 'Three spatial dimensions', 'The red, green, and blue channels', 'Three different images'], 2], ['What does `image[::2, ::2]` do to a 2-D image array?', ['It halves the bit depth', 'It retains every second row and every second column', 'It doubles the image size', 'It converts the image to grayscale'], 1], ['Why is it often useful to convert an integer image to floating point before arithmetic?', ['Floating point images always use less memory', 'It avoids integer range/overflow issues and makes numerical operations easier to interpret', 'NumPy cannot perform arithmetic on integers', 'It automatically sharpens the image'], 1], ['Which NumPy operation is preferred for solving the linear system `Ax=b`?', ['`np.linalg.inv(A) @ b`', '`np.linalg.solve(A, b)`', '`A * b`', '`np.linalg.norm(A-b)`'], 1], ['What does a norm provide in the context of comparing two images?', ['A color conversion', 'A single non-negative number measuring the size of their difference', 'The number of pixels in the image', 'A new sampling grid'], 1]]


In [ ]:
import ipywidgets as widgets
from IPython.display import display

radio_widgets = []
output = widgets.Output()

quiz_boxes = []
for i, (question, options, correct) in enumerate(QUESTIONS):
    label = widgets.HTML(f"<b>Q{i+1}.</b> {question}")
    radio = widgets.RadioButtons(options=options, description='', disabled=False,
                                   layout=widgets.Layout(width='max-content'))
    radio_widgets.append(radio)
    quiz_boxes.append(widgets.VBox([label, radio]))

check_button = widgets.Button(description='Check answers', button_style='success')

def check_answers(b):
    output.clear_output()
    with output:
        score = 0
        for i, (question, options, correct) in enumerate(QUESTIONS):
            chosen = radio_widgets[i].value
            chosen_index = options.index(chosen) if chosen in options else -1
            if chosen_index == correct:
                score += 1
                print(f"Q{i+1}: Correct!")
            else:
                print(f"Q{i+1}: Not quite. Correct answer: {options[correct]}")
        print(f"\nScore: {score} / {len(QUESTIONS)}")

check_button.on_click(check_answers)

display(widgets.VBox(quiz_boxes + [check_button, output]))